# 03. 로컬 판단 모델(tev1) 사전 실험
**왜 로컬 모델인가**: 센터 데이터는 외부로 보낼 수 없고(게시판 답변), 외부 LLM API는 비용이 든다 → 센터 PC 안에서 무료로 도는 판단 모델을 기본으로 쓴다.

**tev1**: Together AI의 판단(decision) 모델(Qwen3.5 미세조정, 0.8B·4B), Ollama `/v1/systemone`. 보기 중 고르기·참거짓·등급 + **보기별 확률**을 돌려준다. 라이선스 Apache 2.0(모델)·MIT(open-jev).

**실험**: 정답을 정해 둔 한국어 원인 판정 10건(`eval/tev1_probe.py`) — 기상·연휴·행사·관광 수요·데이터 오류·판단 불가. CPU 측정은 GPU를 숨긴 별도 Ollama 서버로 했다(요청 옵션 `num_gpu:0`은 이 API에서 무시됨).

In [1]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
pd.set_option("display.max_colwidth", 60)
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False

In [2]:
import json
res = []
for p in sorted(Path("eval/results").glob("tev1_probe_*.json")):
    j = json.loads(p.read_text("utf-8"))
    sm = j["summary"]
    res.append({"모델": sm["model"], "장치": "CPU만" if sm["cpu_only"] else "GPU", "정답률": sm["accuracy"],
                "첫 호출(초)": sm["sec_first"], "1건당 중앙값(초)": sm["sec_median_rest"], "파일": p.name})
pd.DataFrame(res)

,모델,장치,정답률,첫 호출(초),1건당 중앙값(초),파일
0,tev1:0.8b,CPU만,0.5,2.00,0.68,tev1_probe_tev1_0.8b_cpu.json
1,tev1:0.8b,GPU,0.6,41.06,0.03,tev1_probe_tev1_0.8b_gpu.json
2,tev1:4b,CPU만,1.0,7.40,4.12,tev1_probe_tev1_4b_cpu.json
3,tev1:4b,GPU,1.0,2.50,0.06,tev1_probe_tev1_4b_gpu.json


※ `_cpu` 결과는 GPU를 숨긴 별도 Ollama 서버(포트 11501)에서 잰 값, `_gpu`는 RTX 4070 Ti에서 잰 값이다.

In [3]:
j = json.loads(Path("eval/results/tev1_probe_tev1_4b_cpu.json").read_text("utf-8"))
pd.DataFrame([{"정답": r["gold"], "예측": r["pred"], "확신도": round(r["confidence"], 2), "초": r["seconds"], "사례": r["state"][:50]} for r in j["rows"]])

,정답,예측,확신도,초,사례
0,tourism_demand,tourism_demand,0.71,7.40,"성산읍 방문인구 41% 증가. 같은 날 버스 승차 +35%, 성산일출봉 와이파이 이용자 +"
1,weather,weather,0.88,4.21,"애월읍 방문인구 38% 감소. 같은 날 애월 관측소 일강수량 112mm, 최대풍속 21m/"
2,holiday,holiday,0.83,3.95,구좌읍 방문인구 47% 증가. 이날은 설날 연휴 이틀째. 강수 없음. 행사 없음.
3,event,event,0.97,4.13,표선면 방문인구 60% 증가. 같은 날 '표선해변 하얀모래 축제' 진행 중(표선면). 날씨
4,data_issue,data_issue,0.82,3.98,연동 방문인구가 0으로 기록됨. 같은 날 버스 승차와 와이파이 이용은 평소 수준. 이 날짜
5,unknown,unknown,0.90,3.90,중문동 와이파이 이용자 25% 감소. 다른 신호는 평소 수준. 날씨·공휴일·행사 정보 없음
6,weather,weather,0.95,4.12,"한림읍 버스 승차 44% 감소. 한림 관측소 최대풍속 26m/s, 태풍 영향권. 방문인구도"
7,holiday,holiday,0.92,3.93,"우도면 방문인구 55% 증가, 와이파이 +61%. 추석 연휴 첫날. 강수 0mm."
8,event,event,0.97,4.16,건입동 방문인구 33% 증가. '탐라문화제' 개최(건입동 일대). 같은 날 버스 승차 +2
9,data_issue,data_issue,0.73,4.16,"노형동 버스 승차가 평소의 4배로 기록됨. 해당 날짜 정류소 1곳의 값만 비정상적으로 큼,"


**결론**: 4B를 기본 판단 모델로 채택(GPU·CPU 모두 10/10, CPU 1건 약 4초). 0.8B는 '판단 불가'로 쏠려 부적합.
**한계**: 단서가 분명한 손수 만든 10건 → 실제 성능은 가짜 이상 주입 평가에서 규칙 기반·외부 LLM과 비교해 다시 잰다. 센터 PC(i7-8700) 속도는 미확인.